# Deep Learning Layer Subclassing: PyTorch nn.Module & Keras Architecture

**Session:** 2026-11-01 · OOP with Python  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking

Modern deep learning frameworks—most notably **PyTorch** and **Keras**—are built entirely upon Object-Oriented Programming (OOP) paradigms. Rather than defining neural networks as static graphs or flat function pipelines, deep learning models are constructed by **subclassing base module abstractions**:

1. In **PyTorch**, every custom layer and neural network subclasses `torch.nn.Module`. Subclasses initialize submodules in `__init__()` (calling `super().__init__()`) and override the computational graph in `forward()`.
2. In **Keras**, custom layers subclass `tf.keras.layers.Layer`, implementing state initialization in `__init__()`, dynamic weight allocation in `build(input_shape)`, and computational execution in `call(inputs)`.

In this notebook, we dissect the architectural design of deep learning subclassing, build a zero-dependency layer hierarchy from scratch using NumPy, and implement a custom Multi-Layer Perceptron (MLP) for Fashion-MNIST classification.

## Learning Outcomes

- Understand why object-oriented subclassing is the standard paradigm in modern deep learning frameworks.
- Understand the responsibilities of `__init__()`, `build()`, and `call()` / `forward()` in neural layer architectures.
- Implement a pure NumPy `Layer` base class with automatic parameter tracking and callable instance dispatch (`__call__`).
- Construct custom `Dense`, `ReLU`, and `Softmax` layers subclassing the common base abstraction.
- Assemble and execute a complete 784-128-10 Multi-Layer Perceptron (MLP) for Fashion-MNIST image classification.

## 1. Architectural Paradigms: PyTorch vs Keras Subclassing

Both major deep learning ecosystems converge on class inheritance to model neural transformations:

| Framework | Base Class | State Initialization | Weight Allocation | Forward Computation |
| :--- | :--- | :--- | :--- | :--- |
| **PyTorch** | `torch.nn.Module` | `__init__()` with `super().__init__()` | In `__init__()` via `nn.Parameter` | `forward(*inputs)` |
| **Keras** | `tf.keras.layers.Layer` | `__init__()` with `super().__init__()` | In `build(input_shape)` via `self.add_weight()` | `call(inputs)` |

### The Value of Callable Instances (`__call__`)
In both frameworks, users never invoke `model.forward(x)` or `layer.call(x)` directly. Instead, they invoke the object like a function:
```python
output = model(input_tensor)
```
Behind the scenes, Python dispatches `model(x)` to the dunder method `__call__(self, *args, **kwargs)`. The base class `Module` or `Layer` uses `__call__` to run pre-forward hooks, track computational graphs, invoke `forward()` / `call()`, and execute post-forward hooks.

## 2. Building a Deep Learning Layer Hierarchy from First Principles

To understand deep learning subclassing intimately without framework dependencies, we implement an object-oriented layer hierarchy using pure NumPy.

### Mathematical Formulation of Dense Layer
For an input tensor $\mathbf{X} \in \mathbb{R}^{B \times D_{\text{in}}}$ with batch size $B$ and feature dimension $D_{\text{in}}$, the linear affine transformation produces logits $\mathbf{Z} \in \mathbb{R}^{B \times D_{\text{out}}}$:

$$\mathbf{Z} = \mathbf{X} \mathbf{W} + \mathbf{b}$$

where $\mathbf{W} \in \mathbb{R}^{D_{\text{in}} \times D_{\text{out}}}$ is the weight matrix initialized via He (Kaiming) normal initialization:
$$\mathbf{W} \sim \mathcal{N}\left(0, \sqrt{\frac{2}{D_{\text{in}}}}\right)$$
and $\mathbf{b} \in \mathbb{R}^{D_{\text{out}}}$ is the bias vector initialized to zeros.

In [ ]:
from abc import ABC, abstractmethod

import numpy as np


class Layer(ABC):
    """Abstract base class for all neural network layers and modules."""

    def __init__(self, name: str | None = None) -> None:
        self.name = name or self.__class__.__name__
        self._parameters: dict[str, np.ndarray] = {}
        self._sublayers: dict[str, Layer] = {}
        self.is_trainable: bool = True

    def register_parameter(self, param_name: str, tensor: np.ndarray) -> None:
        """Register a trainable parameter array in the module namespace."""
        self._parameters[param_name] = tensor

    def register_sublayer(self, layer_name: str, layer: "Layer") -> None:
        """Register a child layer component."""
        self._sublayers[layer_name] = layer

    @property
    def total_parameters(self) -> int:
        """Count total scalar parameters across this layer and nested sublayers."""
        direct_params = sum(p.size for p in self._parameters.values())
        sublayer_params = sum(layer.total_parameters for layer in self._sublayers.values())
        return direct_params + sublayer_params

    def __call__(self, inputs: np.ndarray) -> np.ndarray:
        """Callable instance protocol wrapping forward pass computation."""
        return self.forward(inputs)

    @abstractmethod
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """Compute the forward transformation on input tensors."""
        pass

    def __repr__(self) -> str:
        return f"<{self.name} | Parameters: {self.total_parameters:,}>"


print("Base Layer abstraction successfully defined.")

## 3. Concrete Layer Implementations: `Dense`, `ReLU`, and `Softmax`

Now we subclass `Layer` to create our foundational neural network building blocks:
1. `Dense`: Fully connected linear layer with He normal initialization.
2. `ReLU`: Rectified Linear Unit element-wise activation:
   $$\text{ReLU}(x) = \max(0, x)$$
3. `Softmax`: Probability normalization layer with numerical stability subtraction:
   $$\sigma(\mathbf{z})_i = \frac{e^{z_i - \max(\mathbf{z})}}{\sum_{j} e^{z_j - \max(\mathbf{z})}}$$

In [ ]:
class Dense(Layer):
    """Fully connected affine layer: Z = X W + b."""

    def __init__(self, in_features: int, out_features: int, name: str | None = None) -> None:
        super().__init__(name=name)
        self.in_features = in_features
        self.out_features = out_features

        # He (Kaiming) normal weight initialization
        std = np.sqrt(2.0 / in_features)
        weights = np.random.randn(in_features, out_features) * std
        biases = np.zeros(out_features)

        self.register_parameter("weights", weights)
        self.register_parameter("biases", biases)

    def forward(self, inputs: np.ndarray) -> np.ndarray:
        w = self._parameters["weights"]
        b = self._parameters["biases"]
        return np.dot(inputs, w) + b


class ReLU(Layer):
    """Element-wise Rectified Linear Unit activation."""

    def forward(self, inputs: np.ndarray) -> np.ndarray:
        return np.maximum(0.0, inputs)


class Softmax(Layer):
    """Numerically stable softmax probability distribution."""

    def forward(self, inputs: np.ndarray) -> np.ndarray:
        # Subtract max per row for numerical stability
        shifted = inputs - np.max(inputs, axis=-1, keepdims=True)
        exp_vals = np.exp(shifted)
        return exp_vals / np.sum(exp_vals, axis=-1, keepdims=True)


class Sequential(Layer):
    """Container executing an ordered sequence of child layers."""

    def __init__(self, *layers: Layer, name: str | None = None) -> None:
        super().__init__(name=name)
        self.layers: list[Layer] = list(layers)
        for idx, layer in enumerate(self.layers):
            self.register_sublayer(f"layer_{idx}_{layer.name}", layer)

    def forward(self, inputs: np.ndarray) -> np.ndarray:
        out = inputs
        for layer in self.layers:
            out = layer(out)
        return out

## 4. Case Study: Fashion-MNIST MLP Classification Model

We now implement the Fashion-MNIST neural network from the lecture slides:
- **Input:** $28 \times 28$ grayscale pixel images flattened into a 784-dimensional vector.
- **Hidden Layer:** Dense layer projecting from 784 nodes down to 128 nodes, followed by ReLU activation.
- **Output Layer:** Dense layer projecting from 128 nodes down to 10 class logits, followed by Softmax activation.
- **Output:** 10-dimensional probability vector corresponding to the 10 Fashion-MNIST clothing categories (T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot).

In [ ]:
class FashionMNISTMLP(Layer):
    """Subclassed Multi-Layer Perceptron for Fashion-MNIST classification."""

    def __init__(self) -> None:
        super().__init__(name="FashionMNIST_MLP_Classifier")

        self.flatten_dim = 28 * 28  # 784 input dimensions
        self.hidden_dim = 128
        self.num_classes = 10

        # Submodules registered within the model
        self.fc1 = Dense(self.flatten_dim, self.hidden_dim, name="fc1_dense")
        self.act1 = ReLU(name="relu_act")
        self.fc2 = Dense(self.hidden_dim, self.num_classes, name="fc2_logits")
        self.classifier = Softmax(name="softmax_out")

        # Register submodules for parameter tracking
        self.register_sublayer("fc1", self.fc1)
        self.register_sublayer("act1", self.act1)
        self.register_sublayer("fc2", self.fc2)
        self.register_sublayer("classifier", self.classifier)

    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """Forward pass implementation."""
        # Flatten inputs if shape is (Batch, 28, 28)
        if inputs.ndim == 3 and inputs.shape[1:] == (28, 28):
            inputs = inputs.reshape(inputs.shape[0], -1)

        x = self.fc1(inputs)
        x = self.act1(x)
        logits = self.fc2(x)
        probabilities = self.classifier(logits)
        return probabilities


# Instantiate model
np.random.seed(42)
model = FashionMNISTMLP()

print("=" * 60)
print(f"Model Architecture: {model.name}")
print(f"Total Model Parameters: {model.total_parameters:,}")
print("=" * 60)
for name, sublayer in model._sublayers.items():
    print(f"  [{name:12s}] -> {sublayer}")

## 5. Forward Pass and Inference Simulation

Let's simulate a batch of 4 Fashion-MNIST images ($B = 4$, shape $(4, 28, 28)$), run the forward inference pass through our subclassed model, and inspect the resulting probability distributions and predicted class labels.

In [ ]:
# Fashion-MNIST category labels:
CLASS_LABELS = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

# Create synthetic input batch of 4 images:
batch_size = 4
synthetic_images = np.random.uniform(0.0, 1.0, size=(batch_size, 28, 28))

# Execute forward inference using the __call__ protocol:
predicted_probabilities = model(synthetic_images)
predicted_classes = np.argmax(predicted_probabilities, axis=1)

print(f"Input batch shape:         {synthetic_images.shape}")
print(f"Output probabilities shape: {predicted_probabilities.shape}\n")

print(f"{'Sample':8s} | {'Top Predicted Class':20s} | {'Confidence':12s} | {'Probability Sum':16s}")
print("-" * 65)
for i in range(batch_size):
    cls_idx = predicted_classes[i]
    cls_name = CLASS_LABELS[cls_idx]
    confidence = predicted_probabilities[i, cls_idx]
    prob_sum = np.sum(predicted_probabilities[i])
    print(f"Item #{i + 1:<4d} | {cls_name:20s} | {confidence:10.4f}   | {prob_sum:14.6f}")

# Verify probabilities sum to 1.0 for each sample in the batch:
assert np.allclose(np.sum(predicted_probabilities, axis=1), 1.0)
print("\nAll probability invariants mathematically verified.")

## 6. PyTorch Equivalent Architecture Reference

For environments with PyTorch installed, here is how the exact same model translates to PyTorch's `nn.Module` subclassing convention:

```python
import torch
import torch.nn as nn


class PyTorchFashionMNIST(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
            nn.Softmax(dim=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        flat = self.flatten(x)
        return self.network(flat)
```

Notice the direct 1-to-1 structural mapping between our NumPy implementation and the PyTorch framework:
- `super().__init__()` registers internal parameter tracking dictionaries.
- Submodules assigned as attributes are automatically traversed.
- The `forward()` method defines the computational dataflow graph.